##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Audio Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Audio.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook provides an example of how to prompt Gemini Flash using an audio file. In this case, you'll use a [sound recording](https://www.jfklibrary.org/asset-viewer/archives/jfkwha-006) of President John F. Kennedy’s 1961 State of the Union address.

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Audio.ipynb).

## Setup

### Install dependencies

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.0 for Interactions API

### Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [10]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

Select the model you want to use in this guide:

In [12]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-3.6-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

### Upload an audio file with the File API

To use an audio file in your prompt, you must first upload it using the [File API](../quickstarts/File_API.ipynb).


In [14]:
URL = "https://storage.googleapis.com/generativeai-downloads/data/State_of_the_Union_Address_30_January_1961.mp3"

In [15]:
!wget -q $URL -O sample.mp3

In [16]:
your_audio_file = client.files.upload(file='sample.mp3')

## Use the file in your prompt

In [18]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Listen carefully to the following audio file. Provide a brief summary."},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

print(interaction.steps[-1].content[0].text)

In his first State of the Union address on January 30, 1961, President John F. Kennedy presents a candid assessment of a nation facing both "national peril and national opportunity." He details a struggling domestic economy marked by recession, high unemployment, and stagnant growth, proposing a series of legislative measures—including minimum wage increases, urban redevelopment, and expanded unemployment benefits—to stimulate recovery. 

On the international front, Kennedy addresses the heightening tensions of the Cold War, highlighting crises in Asia, Africa, and Latin America. He calls for a significant strengthening of U.S. military capabilities, specifically through increased airlift capacity and accelerated missile and submarine programs. Simultaneously, he champions a proactive foreign policy focused on international development, proposing the "Alliance for Progress" for Latin America and the creation of the Peace Corps. Kennedy concludes by emphasizing the need for scientific c

## Inline Audio

For small requests you can inline the audio data into the request, like you can with images. The audio data needs to be encoded in base64 to be sent with the prompt. Use for example PyDub to trim the first 10s of the audio:

In [ ]:
%pip install -Uq pydub

In [22]:
from pydub import AudioSegment

ModuleNotFoundError: No module named 'pyaudioop'

In [23]:
sound = AudioSegment.from_mp3("sample.mp3")

NameError: name 'AudioSegment' is not defined

In [24]:
sound[:10000] # slices are in ms

NameError: name 'sound' is not defined

Add it to the list of parts in the prompt:

In [26]:
import base64

audio_data = base64.b64encode(sound[:10000].export().read()).decode('utf-8')

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Describe this audio clip"},
        {"type": "audio", "data": audio_data, "mime_type": "audio/mp3"},
    ],
)

print(interaction.steps[-1].content[0].text)

NameError: name 'sound' is not defined

Note the following about providing audio as inline data:

- The maximum request size is 100 MB, which includes text prompts, system instructions, and files provided inline. If your file's size will make the total request size exceed 100 MB, then [use the File API](https://ai.google.dev/gemini-api/docs/audio?lang=python#upload-audio) to upload files.
- If you're using an audio sample multiple times, it is more efficient to [use the File API](https://ai.google.dev/gemini-api/docs/audio?lang=python#upload-audio).


## Get a transcript of the audio file
To get a transcript, just ask for it in the prompt. For example:


In [29]:
prompt = "Generate a transcript of the speech."

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

from IPython.display import Markdown
display(Markdown(interaction.steps[-1].content[0].text))

### Refer to timestamps in the audio file
A prompt can specify timestamps of the form `MM:SS` to refer to particular sections in an audio file. For example:

In [31]:
# Create a prompt containing timestamps.
prompt = "Provide a transcript of the speech between the timestamps 02:30 and 03:29."

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

display(Markdown(interaction.steps[-1].content[0].text))

Here's a transcript of the speech between the timestamps 02:30 and 03:29:

To be back among so many friends is a happy one. I am confident that that friendship will continue. Our Constitution wisely assigns both joint and separate roles to each branch of the Government; and a President and a Congress who hold each other in mutual respect will neither permit nor attempt any trespass. For my part, I shall withhold from neither the Congress nor the people any fact or report, past, present, or future, which is necessary for an informed judgment of our conduct and hazards. I shall neither shift the burden of executive decisions to the Congress, nor avoid responsibility for the outcome of those decisions.


## Use a YouTube video

In [33]:
from IPython.display import display, Markdown

youtube_url = "https://www.youtube.com/watch?v=RDOMKIw1aF4" # @param {type:"string"}

prompt = """
    Analyze the following YouTube video content. Provide a concise summary covering:

    1.  **Main Thesis/Claim:** What is the central point the creator is making?
    2.  **Key Topics:** List the main subjects discussed, referencing specific examples or technologies mentioned (e.g., AI models, programming languages, projects).
    3.  **Call to Action:** Identify any explicit requests made to the viewer.
    4.  **Summary:** Provide a concise summary of the video content.

    Use the provided title, chapter timestamps/descriptions, and description text for your analysis.
"""
# Analyze the video
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "video", "uri": youtube_url},
    ],
)
display(Markdown(interaction.steps[-1].content[0].text))

<IPython.core.display.Markdown object>


## Count audio tokens

You can count the number of tokens in your audio file using the [count_tokens](https://googleapis.github.io/python-genai/#count-tokens-and-compute-tokens) method.

Audio files have a fixed per second token rate (more details in the dedicated [count token quickstart](./Counting_Tokens.ipynb)).

In [35]:
count_tokens_response = client.models.count_tokens(
    model=MODEL_ID,
    contents=[your_audio_file],
)

print("Audio file tokens:",count_tokens_response.total_tokens)

Audio file tokens: 83528


## Next Steps
### Useful API references:

More details about Gemini API's [vision capabilities](https://ai.google.dev/gemini-api/docs/vision) in the documentation.

If you want to know about the File API, check its [API reference](https://ai.google.dev/api/files) or the [File API](https://github.com/google-gemini/cookbook/blob/main/quickstarts/File_API.ipynb) quickstart.

### Related examples

Check this example using the audio files to give you more ideas on what the gemini API can do with them:
* Share [Voice memos](https://github.com/google-gemini/cookbook/blob/main/examples/Voice_memos.ipynb) with Gemini API and brainstorm ideas

### Continue your discovery of the Gemini API

Have a look at the [Audio](../quickstarts/Audio.ipynb) quickstart to learn about another type of media file, then learn more about [prompting with media files](https://ai.google.dev/tutorials/prompting_with_media) in the docs, including the supported formats and maximum length for audio files. .
